# 🛠️ Environment Setup

This project requires a specific environment to ensure compatibility with **PyTorch**. Please verify that your local environment matches the specifications below.

* **Python** `3.10`
* **PyTorch** `2.7.0`
* **torchaudio** `2.7.0`
* **torchmetrics** `2.6.1`
* **torch-geometric** `2.6.1`


### 🚀 Installation Guide

We recommend using a virtual environment (e.g., Conda or venv) to avoid dependency conflicts. You can install all necessary packages using the provided `requirements.txt` file:

```bash
pip install -r requirements.txt

## 🧩 Supernode Generation Pipeline

To construct a robust graph representation from individual patches, we implement the `supernode_generation` module. This component is designed to aggregate coherent **Super Patches** and filter out topological anomalies.

The generation process follows a strict three-step protocol to ensure both feature consistency and spatial coherence:

1.  **Super Patch Formation**: 
    * The feature similarity must meet `cosine_matrix >= threshold`.
    * The spatial distance is restricted by `euclidean_distances <= spatial_threshold1`.

2.  **Graph Connectivity**: 
    * We establish connections between these super patches to form the global structure.
    * This is governed by the inter-node distance limit `spatial_threshold2`.

3.  **Topology Sanitation (`false_graph_filtering`)**: 
    * Finally, we apply a filtering function to prune unrealistic conditions.
    * This step specifically removes non-connected subgraphs (small components) to maintain graph integrity.


Taking the sample `201334728_HE_8` as an instance:

```text
# 1. Graph Structure (PyTorch Geometric)
# Saves the processed graph topology and features
data/superpatch/201334728_HE_8_0.7_graph_torch.pt

# 2. Construction Metadata (CSV)
# Contains detailed artifact statistics and sophisticated features
data/superpatch/201334728_HE_8_0.7_1100.8_artifact_sophis_final.csv

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import torch
from torch_geometric.data import Data
import pickle
import argparse
import h5py

def supernode_generation(image, Argument, save_dir):

	if os.path.exists(save_dir) is False:
		os.mkdir(save_dir)

	origin_dir = os.path.join(save_dir, 'original')
	if os.path.exists(origin_dir) is False:
		os.mkdir(origin_dir)

	superpatch_dir = os.path.join(save_dir, 'superpatch')
	if os.path.exists(superpatch_dir) is False:
		os.mkdir(superpatch_dir)

	gridNum = Argument.gridNum
	threshold = Argument.threshold
	spatial_threshold1 = Argument.spatial_threshold1*Argument.imagesize
	spatial_threshold2 = Argument.spatial_threshold2*Argument.imagesize

	sample = image.split('/')[-1].split('.')[0]
	sample = image

	save_dir = os.path.join(superpatch_dir, sample+ '_' + str(threshold) + '_graph_torch.pt')
	h5_path = os.path.join(Argument.h5_path, str(sample) + '.h5')

	with h5py.File(h5_path, 'r') as file:
		coords = file['coords'][()]      # shape: [N, 2]
		features = file['features'][()]  # shape: [N, D]

	num_total = features.shape[0]
	num_sample = max(1, num_total // 2)

	sample_indices = np.random.choice(num_total, num_sample, replace=False)

	sampled_features = features[sample_indices]  # shape: [num_sample, D]
	sampled_coords = coords[sample_indices]      # shape: [num_sample, 2]

	feature_list = sampled_features.tolist()
	x_y_list = sampled_coords
	y = x_y_list[:, 1]
	x = x_y_list[:, 0]

	feature_df = pd.DataFrame.from_dict(feature_list)
	coordinate_df = pd.DataFrame({'X': x,'Y': y})	# shape[0]=x shape[1]=y
	graph_dataframe = pd.concat([coordinate_df, feature_df], axis = 1)
	graph_dataframe = graph_dataframe.sort_values(by = ['Y', 'X'])
	graph_dataframe = graph_dataframe.reset_index(drop = True)
	coordinate_df = graph_dataframe.iloc[:,0:2]
	coordinate_df.to_csv(os.path.join(origin_dir, sample+'_node_location_list.csv'))
	index = list(graph_dataframe.index)
	graph_dataframe.insert(0,'index_orig', index)
	node_dict = {}
	
	for i in range(len(coordinate_df)):
		node_dict.setdefault(i,[])
	
	X = max(set(np.squeeze(graph_dataframe.loc[:, ['X']].values,axis = 1)))		# 174864
	Y = max(set(np.squeeze(graph_dataframe.loc[:, ['Y']].values, axis = 1)))	# 196384
	del feature_df
	gridNum = gridNum
	X_size = int(X / gridNum)
	Y_size = int(Y / gridNum)

	with tqdm(total=(gridNum+2)*(gridNum+2)) as pbar:
		for p in range(gridNum+2):
			for q in range(gridNum+2):
				if p == 0 :
					if q == 0:
						is_X = graph_dataframe['X'] <= X_size * (p+1)
						is_X2 = graph_dataframe['X'] >= 0
						is_Y = graph_dataframe['Y'] <= Y_size * (q+1)
						is_Y2 = graph_dataframe['Y'] >= 0
						X_10 = graph_dataframe[is_X & is_Y & is_X2 & is_Y2]
						
					elif q == (gridNum+1):
						is_X = graph_dataframe['X'] <= X_size * (p+1)
						is_X2 = graph_dataframe['X'] >= 0
						is_Y = graph_dataframe['Y'] <= Y
						is_Y2 = graph_dataframe['Y'] >= (Y_size * (q) -2)
						X_10 = graph_dataframe[is_X & is_Y & is_X2 & is_Y2]
						
					else:
						is_X = graph_dataframe['X'] <= X_size * (p+1)
						is_X2 = graph_dataframe['X'] >= 0
						is_Y = graph_dataframe['Y'] <= Y_size * (q+1)
						is_Y2 = graph_dataframe['Y'] >= (Y_size * (q) -2)
						X_10 = graph_dataframe[is_X & is_Y & is_X2 & is_Y2]
				elif p == (gridNum+1) :
					if q == 0:
						is_X = graph_dataframe['X'] <= X
						is_X2 = graph_dataframe['X'] >= (X_size *(p) - 2)
						is_Y = graph_dataframe['Y'] <= Y_size * (q+1)
						is_Y2 = graph_dataframe['Y'] >= 0
						X_10 = graph_dataframe[is_X & is_Y & is_X2 & is_Y2]
					elif q == (gridNum+1):
						is_X = graph_dataframe['X'] <= X
						is_X2 = graph_dataframe['X'] >= (X_size *(p) - 2)
						is_Y = graph_dataframe['Y'] <= Y
						is_Y2 = graph_dataframe['Y'] >= (Y_size * (q) -2)
						X_10 = graph_dataframe[is_X & is_Y & is_X2 & is_Y2]
					else:
						is_X = graph_dataframe['X'] <= X
						is_X2 = graph_dataframe['X'] >= (X_size *(p) - 2)
						is_Y = graph_dataframe['Y'] <= Y_size * (q+1)
						is_Y2 = graph_dataframe['Y'] >= (Y_size * (q) -2)
						X_10 = graph_dataframe[is_X & is_Y & is_X2 & is_Y2]
				else :
					if q == 0:
						is_X = graph_dataframe['X'] <= X_size * (p+1)
						is_X2 = graph_dataframe['X'] >= (X_size *(p) - 2)
						is_Y = graph_dataframe['Y'] <= Y_size * (q+1)
						is_Y2 = graph_dataframe['Y'] >= 0
						X_10 = graph_dataframe[is_X & is_Y & is_X2 & is_Y2]
					elif q == (gridNum+1):
						is_X = graph_dataframe['X'] <= X_size * (p+1)
						is_X2 = graph_dataframe['X'] >= (X_size *(p) - 2)
						is_Y = graph_dataframe['Y'] <= Y
						is_Y2 = graph_dataframe['Y'] >= (Y_size * (q) -2)
						X_10 = graph_dataframe[is_X & is_Y & is_X2 & is_Y2]
					else:
						is_X = graph_dataframe['X'] <= X_size * (p+1)
						is_X2 = graph_dataframe['X'] >= (X_size *(p) - 2)
						is_Y = graph_dataframe['Y'] <= Y_size * (q+1)
						is_Y2 = graph_dataframe['Y'] >= (Y_size * (q) -2)
						X_10 = graph_dataframe[is_X & is_Y & is_X2 & is_Y2]
				if len(X_10) == 0:
					pbar.update()
					continue
				coordinate_dataframe = X_10.loc[:, ['X','Y']]
				X_10 = X_10.reset_index(drop = True)
				coordinate_list = coordinate_dataframe.values.tolist()
				index_list = coordinate_dataframe.index.tolist()
				feature_dataframe = X_10[X_10.columns.difference(['index_orig','X','Y'])]
				feature_list = feature_dataframe.values.tolist()
				coordinate_matrix = euclidean_distances(coordinate_list, coordinate_list)

				coordinate_matrix = np.where(coordinate_matrix > spatial_threshold1, 0 , 1)		
				
				cosine_matrix = cosine_similarity(feature_list, feature_list)
				Adj_list = (coordinate_matrix == 1).astype(int) * (cosine_matrix >= threshold).astype(int)

				for c, item in enumerate(Adj_list):
					for node_index in np.array(index_list)[item.astype('bool')]:
						if node_index == index_list[c]:
							pass
						else:
							node_dict[index_list[c]].append(node_index)
				pbar.update()
	a_file = open(os.path.join(origin_dir, sample + '_node_dict.pkl'), "wb")
	pickle.dump(node_dict, a_file)
	a_file.close()
	dict_len_list = []
	
	for i in range(0, len(node_dict)):
		dict_len_list.append(len(node_dict[i]))
	
	arglist_strict = np.argsort(np.array(dict_len_list))
	arglist_strict = arglist_strict[::-1]

	for arg_value in arglist_strict: 
		if arg_value in node_dict.keys():
			for adj_item in node_dict[arg_value]:
				if adj_item in node_dict.keys():
					node_dict.pop(adj_item)
					arglist_strict=np.delete(arglist_strict, np.argwhere(arglist_strict == adj_item))
	for key_value in node_dict.keys():  
		node_dict[key_value] = list(set(node_dict[key_value]))

	supernode_coordinate_x_strict = []
	supernode_coordinate_y_strict = []
	supernode_feature_strict = []
	
	supernode_relate_value = [supernode_coordinate_x_strict,
							   supernode_coordinate_y_strict,
							   supernode_feature_strict]
	
	whole_feature = graph_dataframe[graph_dataframe.columns.difference(['index_orig','X','Y'])]
	with tqdm(total = len(node_dict.keys())) as pbar_node:
		for key_value in node_dict.keys():
			supernode_relate_value[0].append(graph_dataframe['X'][key_value])
			supernode_relate_value[1].append(graph_dataframe['Y'][key_value])
			if len(node_dict[key_value]) == 0:
				select_feature = whole_feature.iloc[key_value]
			else:
				select_feature = whole_feature.iloc[node_dict[key_value] + [key_value]]
				select_feature = select_feature.mean()
			if len(supernode_relate_value[2]) == 0:
				temp_select = np.array(select_feature)
				supernode_relate_value[2] = np.reshape(temp_select, (1,1536))
			else:
				temp_select = np.array(select_feature)
				supernode_relate_value[2] = np.concatenate((supernode_relate_value[2], np.reshape(temp_select, (1,1536))), axis=0)
			pbar_node.update()

	coordinate_integrate = pd.DataFrame({'X':supernode_relate_value[0],'Y':supernode_relate_value[1]})
	coordinate_matrix1 = euclidean_distances(coordinate_integrate, coordinate_integrate)
	coordinate_matrix1 = np.where(coordinate_matrix1 > spatial_threshold2 , 0 , 1)
	fromlist = []
	tolist = []
	with tqdm(total = len(coordinate_matrix1)) as pbar_pytorch_geom:
		for i in range(len(coordinate_matrix1)):
			temp = coordinate_matrix1[i,:]
			selectindex = np.where(temp > 0)[0].tolist()
			for index in selectindex:
				fromlist.append(int(i))
				tolist.append(int(index))
			pbar_pytorch_geom.update()
	edge_index = torch.tensor([fromlist, tolist], dtype=torch.long)
	x = torch.tensor(supernode_relate_value[2], dtype=torch.float)

	data = Data(x=x, edge_index=edge_index)
	node_dict = pd.DataFrame.from_dict(node_dict, orient='index')
	node_dict.to_csv(os.path.join(superpatch_dir, sample + '_' + str(threshold) + '.csv'))
	print(data)
	torch.save(data, os.path.join(superpatch_dir, sample+ '_' + str(threshold) + '_graph_torch.pt'))
	print('saving in '+str(os.path.join(superpatch_dir, sample+ '_' + str(threshold) + '_graph_torch.pt')))
	
def Parser_main():
	
	parser = argparse.ArgumentParser(description="TEA-graph superpatch generation")
	parser.add_argument("--database", default='FAZJU', help="Use in the savedir", type = str)
	parser.add_argument("--cancertype",default='HCC',help="cancer type",type=str)
	parser.add_argument("--graphdir",default="data/",help="graph save dir",type=str)
	parser.add_argument("--imagedir",default="data/",help="svs file location",type=str)
	parser.add_argument("--h5_path",default="data/",help="svs file location",type=str)
	parser.add_argument("--weight_path",default=None,help="pretrained weight path",type=str)
	parser.add_argument("--threshold", default = 0.7, help = "cosine similarity threshold", type = float)
	parser.add_argument("--spatial_threshold1", default = 2.9, type = float)	
	parser.add_argument("--spatial_threshold2", default = 5.5, type = float)	
	parser.add_argument("--imagesize", default = 256, help ="crop image size", type = int)
	parser.add_argument("--gridNum", default = 4, help ="crop image size", type = int)
	parser.add_argument("--gpu", default = '0' , help = "gpu device number", type = str)
	return parser.parse_args([])

In [2]:
Argument = Parser_main()
save_dir = Argument.graphdir   
image = '201334728_HE_8'
save_dir = '/data2/jsh/ZJU_temp/data'
supernode_generation(image, Argument, save_dir)

100%|██████████| 11861/11861 [00:00<00:00, 70013.28it/s]


Data(x=[11861, 1536], edge_index=[2, 233057])
saving in /data2/jsh/ZJU_temp/data/superpatch/201334728_HE_8_0.7_graph_torch.pt


In [3]:
import torch
import os
import pandas as pd
import networkx as nx
import torch_geometric.utils as g_util
from torch_geometric.data import Data
import numpy as np
from sklearn.metrics import pairwise_distances
from tqdm import tqdm
from torch_geometric.transforms import Polar
import argparse

pd.options.mode.chained_assignment = None

def false_graph_filtering(distance_thresh, Argument):

	directory = os.path.join(Argument.graphdir, 'superpatch')
	sample_files = [f.split('_{}_graph_torch'.format(Argument.threshold))[0] for f in os.listdir(directory) if f.endswith('.pt')]
	graph_pt_save_dir = os.path.join(Argument.graphdir, 'UNI2', 'pt_files')
	os.makedirs(graph_pt_save_dir, exist_ok=True)
	os.makedirs(os.path.join(Argument.graphdir, 'temp'), exist_ok=True)
	super_root_dir = directory
	original_root_dir =  os.path.join(Argument.graphdir, 'original')
	supernode_files = []
	pt_files = []

	different_value = Argument.threshold
	for sample in sample_files:
		supernode_files.append(os.path.join(super_root_dir, sample+'_'+str(different_value)+'.csv'))
		pt_files.append(os.path.join(super_root_dir, sample+'_'+str(different_value)+'_graph_torch.pt'))

	with tqdm(total=len(pt_files)) as pbar:
		for sample_name, supernode_path, pt in zip(sample_files, supernode_files, pt_files):
	
			location = os.path.join(original_root_dir, sample_name + '_node_location_list.csv')
			graph = torch.load(pt,weights_only=False)
			if os.path.isfile(supernode_path):
				supernode = pd.read_csv(supernode_path)
				location = pd.read_csv(location)
				sample = sample_name.split('_')[0]
				supernode_x = []
				supernode_y = []
				supernode_num = supernode['Unnamed: 0'].tolist()
				supernode_x = location.loc[supernode_num]['X']
				supernode_y = location.loc[supernode_num]['Y']
				coordinate_df = pd.DataFrame({'X': supernode_x, 'Y': supernode_y})
				coordinate_list = np.array(coordinate_df.values.tolist())
				coordinate_matrix = pairwise_distances(coordinate_list, n_jobs=8)
				adj_matrix = np.where(coordinate_matrix >= distance_thresh, 0, 1) 
				Edge_label = np.where(adj_matrix == 1)
				Adj_from = np.unique(Edge_label[0], return_counts=True)
				Adj_to = np.unique(Edge_label[1], return_counts=True)
				Adj_from_singleton = Adj_from[0][Adj_from[1] == 1]
				Adj_to_singleton = Adj_to[0][Adj_to[1] == 1]
				Adj_singleton = np.intersect1d(Adj_from_singleton, Adj_to_singleton)
				coordinate_matrix_modify = coordinate_matrix
				fromlist = Edge_label[0].tolist()
				tolist = Edge_label[1].tolist()
				edge_index = torch.tensor([fromlist, tolist], dtype=torch.long)
	
				graph.edge_index = edge_index
				connected_graph = g_util.to_networkx(graph, to_undirected=True)
				connected_graph = [connected_graph.subgraph(item_graph).copy() for item_graph in
								   nx.connected_components(connected_graph) if len(item_graph) > 150]
				

				connected_graph = [connected_graph.subgraph(item_graph).copy() for item_graph in
								nx.connected_components(connected_graph) if len(item_graph) > 50]    

				connected_graph_node_list = []
				for graph_item in connected_graph:
					connected_graph_node_list.extend(list(graph_item.nodes))

				connected_graph = connected_graph_node_list
				connected_graph = list(connected_graph)
				new_node_order_dict = dict(zip(connected_graph, range(len(connected_graph))))
				new_feature = graph.x[connected_graph]
				new_edge_index = graph.edge_index.numpy()
				new_edge_mask_from = np.isin(new_edge_index[0], connected_graph)
				new_edge_mask_to = np.isin(new_edge_index[1], connected_graph)
				new_edge_mask = new_edge_mask_from * new_edge_mask_to
				new_edge_index_from = new_edge_index[0]
				new_edge_index_from = new_edge_index_from[new_edge_mask]
				new_edge_index_from = [new_node_order_dict[item] for item in new_edge_index_from]
				new_edge_index_to = new_edge_index[1]
				new_edge_index_to = new_edge_index_to[new_edge_mask]
				new_edge_index_to = [new_node_order_dict[item] for item in new_edge_index_to]
				new_edge_index = torch.tensor([new_edge_index_from, new_edge_index_to], dtype=torch.long)
	
				new_supernode = supernode.iloc[connected_graph]
				new_supernode = new_supernode.reset_index()
				new_supernode.to_csv(supernode_path.split('.csv')[0] + '_' + str(distance_thresh) + '_artifact_sophis_final.csv')   #---- save 1
				
				actual_pos = location.iloc[new_supernode['Unnamed: 0']]
				actual_pos_temp = actual_pos
				actual_pos = actual_pos[['X', 'Y']].to_numpy()

				coordinate_integrate = pd.DataFrame({'X':actual_pos_temp['X'],'Y':actual_pos_temp['Y']})
				coordinate_integrate.to_csv(os.path.join(Argument.graphdir, 'temp', '{}.csv'.format(sample_name)), index=False)
				actual_pos = torch.tensor(actual_pos)
				actual_pos = actual_pos.float()
				pos_transfrom = Polar()
				new_graph = Data(x=new_feature, edge_index=new_edge_index, pos=actual_pos * 256.0)
				
				new_graph = pos_transfrom(new_graph)
				
				save_path = os.path.join(graph_pt_save_dir, sample_name+'.pt')
				print(new_graph)
				torch.save(new_graph, save_path)
			pbar.update(1)

def Parser_main():
	parser = argparse.ArgumentParser(description="TEA-graph superpatch generation")
	parser.add_argument("--database", default='FAZJU', help="Use in the savedir", type = str)
	parser.add_argument("--cancertype",default='HCC',help="cancer type",type=str)
	parser.add_argument("--graphdir",default="data/",help="graph save dir",type=str)
	parser.add_argument("--imagedir",default="data/",help="svs file location",type=str)
	parser.add_argument("--h5_path",default="data/",help="svs file location",type=str)
	parser.add_argument("--weight_path",default=None,help="pretrained weight path",type=str)
	parser.add_argument("--threshold", default = 0.7, help = "cosine similarity threshold", type = float)
	parser.add_argument("--spatial_threshold1", default = 2.9, type = float)
	parser.add_argument("--spatial_threshold2", default = 5.5, type = float)
	parser.add_argument("--imagesize", default = 256, help ="crop image size", type = int)
	parser.add_argument("--gridNum", default = 4, help ="crop image size", type = int)
	parser.add_argument("--gpu", default = '0' , help = "gpu device number", type = str)
	return parser.parse_args([])

In [4]:
Argument = Parser_main()
save_dir = '/data'
print(Argument)
false_graph_filtering(4.3*Argument.imagesize, Argument)

Namespace(database='FAZJU', cancertype='HCC', graphdir='/data2/jsh/ZJU_temp/data', imagedir='/data2/jsh/ZJU_temp/data', h5_path='/data2/jsh/ZJU/data_processed/Origin/HE_256/UNI2/h5_files', weight_path=None, threshold=0.7, spatial_threshold1=2.9, spatial_threshold2=5.5, imagesize=256, gridNum=4, gpu='0')


  0%|          | 0/1 [00:00<?, ?it/s]

saving 201334728_HE_8


100%|██████████| 1/1 [00:02<00:00,  2.59s/it]
